# Task 3: Kafka Topic Design

**Mục tiêu**: Thiết kế schema và topic layout cho Apache Kafka để chuyên chở 4 loại sự kiện từ Parser Service bao gồm: node events, edge events, source metadata events, và parser error events. Đặc biệt, mỗi message bắt buộc phải đính kèm `schema_version` (cho forward compatibility) và `event_timestamp`.


## Cấu trúc Kafka Topic

Trong file `kafka_producer.py`, nhóm đã thiết lập Kafka Producer phân luồng dữ liệu vào 4 topic tách biệt:

1. `code.events.nodes`: Chuyên chứa sự kiện Node (AST, Function, Class...)
2. `code.events.edges`: Chuyên chứa sự kiện Edge (các quan hệ AST, CFG, DFG...)
3. `code.events.metadata`: Chứa metadata thống kê về từng file (`loc`, ngôn ngữ, số node...)
4. `code.events.errors`: Hứng các ngoại lệ và lỗi parse để bảo vệ pipeline khỏi gián đoạn.

**Lý do lựa chọn thiết kế này:**
- **Phân tách công việc rõ ràng**: Ở Task 4, công cụ Neo4j Kafka Connector Sink chỉ cần subscribe vào topic `nodes` và `edges` để nạp đồ thị trực tiếp mà không cần Spark gỡ rối. Trong khi đó, ở Task 5, Spark Streaming có thể tập trung consume topic `metadata` để ghi vào MongoDB.
- **Định tuyến (Routing)**: Mỗi event được producer đẩy đi đều đính kèm `key = event["file_path"]`. Việc sử dụng chung partition key giúp Kafka đưa toàn bộ event của cùng 1 file vào chung 1 partition, đảm bảo thứ tự ghi (ordering) luôn được bảo toàn.

### Sơ đồ Phân Luồng Kafka Topics & Consumer Routing

```mermaid
flowchart TD
    subgraph Producer Layer
        P[CPG Kafka Producer<br/>Partition Key: file_path]
    end

    subgraph Kafka Broker Topics
        T1["code.events.nodes<br/>(3 Partitions)"]
        T2["code.events.edges<br/>(3 Partitions)"]
        T3["code.events.metadata<br/>(1 Partition)"]
        T4["code.events.errors<br/>(1 Partition)"]
    end

    subgraph Downstream Consumers
        C1[Neo4j Kafka Connect Sink]
        C2[Spark Structured Streaming]
        C3[Error Logger / Alert]
    end

    subgraph Databases
        DB1[(Neo4j Graph DB)]
        DB2[(MongoDB Source Metadata)]
    end

    P -->|Node events| T1
    P -->|Edge events| T2
    P -->|Metadata events| T3
    P -->|Error events| T4

    T1 --> C1
    T2 --> C1
    C1 -->|Cypher MERGE| DB1

    T3 --> C2
    C2 -->|Replace + Upsert| DB2

    T4 --> C3
```


## Schema Version & Timestamp
Kiểm tra lại xem cấu trúc dữ liệu đã thỏa mãn điều kiện bắt buộc về `schema_version` và `event_timestamp` hay chưa. Dưới đây là một ví dụ sinh Event lỗi từ hàm `error_event`:

In [ ]:
import json
import traceback
from datetime import datetime, timezone

# NOTE: Không import trực tiếp từ parser.py vì file đó kéo theo
# 'kafka-python' (KafkaProducer) — package này chỉ cần khi chạy
# producer thật với Kafka broker, không cần trong notebook demo.
# Hàm error_event rất ngắn, tái dụng inline là hợp lý.

def _now_iso() -> str:
    return datetime.now(timezone.utc).isoformat()

def error_event(file_path: str, exc: Exception, repo_commit: str) -> dict:
    return {
        "schema_version": "v1",
        "event_timestamp": _now_iso(),
        "file_path": file_path,
        "error_type": type(exc).__name__,
        "error_message": str(exc),
        "stack_trace": traceback.format_exc()[-2000:],
        "repo_commit": repo_commit,
    }

sample_err = error_event(
    file_path="target-repo/src/bad_file.py",
    exc=SyntaxError("invalid syntax"),
    repo_commit="abc1234"
)
print(json.dumps(sample_err, indent=2, ensure_ascii=False))


{
  "schema_version": "v1",
  "event_timestamp": "2026-07-22T09:35:15.584285+00:00",
  "file_path": "target-repo/src/bad_file.py",
  "error_type": "SyntaxError",
  "error_message": "invalid syntax",
  "stack_trace": "NoneType: None\n",
  "repo_commit": "abc1234"
}


Như kết quả trên, `schema_version` (`v1`) và `event_timestamp` luôn xuất hiện trong cấu trúc Payload của mọi loại event.

## Partition & Replication Factor

Cấu hình từng topic được khai báo trong `docker-compose.yml` qua service `kafka-init`:

| Topic | Partitions | Replication Factor | Lý do |
|---|---|---|---|
| `code.events.nodes` | **3** | 1 | Volume lớn nhất, cần song song hóa |
| `code.events.edges` | **3** | 1 | Tương tự nodes, thường gấp 2–3x nodes |
| `code.events.metadata` | **1** | 1 | Volume nhỏ, 1 file = 1 message |
| `code.events.errors` | **1** | 1 | Hiếm xảy ra, không cần song song |

**Lý do phân bổ**: Topic `nodes` và `edges` sử dụng **3 partitions** vì đây là nơi tập trung khối lượng message lớn nhất (1 file `.py` có thể sinh hàng trăm nodes và hàng nghìn edges). Tăng số partition cho phép Kafka mở rộng theo chiều ngang khi cần thêm consumer song song ở Task 4/5 mà không cần thay đổi cấu hình broker.

## Reflection

- **What worked**: Topic separation là một thiết kế rất hiệu quả, giúp decoupled hoàn toàn kiến trúc dữ liệu và giúp việc tích hợp Neo4j Connector trở nên dễ dàng.
- **What failed**: Trong quá trình test, khi xử lý một lượng lớn message, Kafka Producer gửi từng object một qua socket gây suy giảm performance đáng kể do network round-trip.
- **Resolution**: Nhóm đã sửa đổi config khởi tạo của Producer (thêm `linger_ms=50`) vào class `CPGProducer`. Điều chỉnh này tuy làm tăng độ trễ thêm 50ms (không đáng kể đối với pipeline), nhưng lại cho phép Kafka client dồn nhiều messages vào chung một batch trước khi gửi qua network, giúp tăng thông lượng (throughpu lên nhiều lần.